<a href="https://colab.research.google.com/github/nihatrasulzada/NVideo-Copilot/blob/main/Grammar_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install transformers datasets torch accelerate sentencepiece

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
dataset = load_dataset("agentlans/grammar-correction")

train_data = dataset['train'].select(range(0, 5000))
val_data = dataset['train'].select(range(5000, 5500))

model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

def preprocess_function(examples):
    inputs = examples['input']
    targets = examples['output']

    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_val = val_data.map(preprocess_function, batched=True, remove_columns=val_data.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-grammar-correction",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

model.save_pretrained("./saved_bart_model")
tokenizer.save_pretrained("./saved_bart_model")
print("Model uğurla yadda saxlanıldı!")

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

train.jsonl.zst: reconstructing file:   0%|          |  0.00B / 6.91MB            

train.jsonl.zst: downloading bytes:           |  0.00B            

validation.jsonl.zst: reconstructing file:   0%|          |  0.00B / 1.77MB            

validation.jsonl.zst: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer
model = BartForConditionalGeneration.from_pretrained("./saved_bart_model")
tokenizer = BartTokenizer.from_pretrained("./saved_bart_model")

In [ ]:

def correct_grammar(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True)
    outputs = model.generate(inputs["input_ids"], max_length=128, num_beams=4, early_stopping=True)
    corrected_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_text

bad_sentence = "How Long Doo You Have to Go To Schools To Be Biochemist?"
result = correct_grammar(bad_sentence)

print("Original:", bad_sentence)
print("Corrected:", result)